In [1]:
!pip install sqlite-utils


  Attempting uninstall: click

    Found existing installation: click 8.2.1

    Uninstalling click-8.2.1:

      Successfully uninstalled click-8.2.1

   ---------- ----------------------------- 1/4 [click]
   ---------- ----------------------------- 1/4 [click]
   ------------------------------ --------- 3/4 [sqlite-utils]
   ---------------------------------------- 4/4 [sqlite-utils]



In [3]:
import sqlite3
import polars as pl
from pathlib import Path

file_loc = './Files/Sample_Superstore.csv'
sqlite_dir = Path('./Files/sqlite/Sample_Superstore')
df = pl.read_csv(file_loc)

In [4]:
if not sqlite_dir.exists():
    sqlite_dir.mkdir(parents=True, exist_ok=True)
sqlite_file = sqlite_dir / 'sample_store.sqlite'

In [6]:
uri = 'sqlite:///' + sqlite_file.as_posix()
uri

'sqlite:///Files/sqlite/Sample_Superstore/sample_store.sqlite'

In [8]:
if sqlite_dir.exists():
    (
        df.sort('Customer_ID')
        .write_database(
            table_name='records',
            connection=uri,
            if_table_exists='replace',
            engine='sqlalchemy'
        )
    )

In [10]:
!pip install adbc-driver-manager

   ---------------------------------------- 0.0/758.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/758.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/758.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/758.9 kB ? eta -:--:--
   --------------------------- ------------ 524.3/758.9 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 758.9/758.9 kB 1.9 MB/s  0:00:00


In [12]:
!pip install adbc-driver-sqlite 

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.7 MB/s  0:00:01

   -------------------- ------------------- 1/2 [adbc-driver-sqlite]
   ---------------------------------------- 2/2 [adbc-driver-sqlite]



In [13]:
dfb = pl.read_database_uri('select * from records limit 3', uri=uri, engine='adbc') 
dfb

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,f64,i64,f64,f64
1160,"""CA-2017-147039""","""29-06-2017""","""04-07-2017""","""Standard Class""","""AA-10315""","""Alex Avila""","""Consumer""","""United States""","""Minneapolis""","""Minnesota""",55407,"""Central""","""OFF-AP-10000576""","""Office Supplies""","""Appliances""","""Belkin 325VA UPS Surge Protect…",362.94,3,0.0,90.735
1161,"""CA-2017-147039""","""29-06-2017""","""04-07-2017""","""Standard Class""","""AA-10315""","""Alex Avila""","""Consumer""","""United States""","""Minneapolis""","""Minnesota""",55407,"""Central""","""OFF-BI-10004654""","""Office Supplies""","""Binders""","""Avery Binding System Hidden Ta…",11.54,2,0.0,5.77
1300,"""CA-2015-121391""","""04-10-2015""","""07-10-2015""","""First Class""","""AA-10315""","""Alex Avila""","""Consumer""","""United States""","""San Francisco""","""California""",94109,"""West""","""OFF-ST-10001590""","""Office Supplies""","""Storage""","""Tenex Personal Project File wi…",26.96,2,0.0,7.0096


In [14]:
dfb = pl.read_database_uri('select * from records', uri=uri, engine='adbc') 
dfb.shape

(9994, 21)

In [17]:
dfb.filter(pl.col('Profit') > 1000).head()

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,f64,i64,f64,f64
354,"""CA-2016-129714""","""01-09-2016""","""03-09-2016""","""First Class""","""AB-10060""","""Adam Bellavance""","""Home Office""","""United States""","""New York City""","""New York""",10009,"""East""","""OFF-BI-10004995""","""Office Supplies""","""Binders""","""GBC DocuBind P400 Electric Bin…",4355.168,4,0.2,1415.4296
9040,"""CA-2016-117121""","""17-12-2016""","""21-12-2016""","""Standard Class""","""AB-10105""","""Adrian Barton""","""Consumer""","""United States""","""Detroit""","""Michigan""",48205,"""Central""","""OFF-BI-10000545""","""Office Supplies""","""Binders""","""GBC Ibimaster 500 Manual ProCl…",9892.74,13,0.0,4946.37
516,"""CA-2017-127432""","""22-01-2017""","""27-01-2017""","""Standard Class""","""AD-10180""","""Alan Dominguez""","""Home Office""","""United States""","""Great Falls""","""Montana""",59405,"""West""","""TEC-CO-10003236""","""Technology""","""Copiers""","""Canon Image Class D660 Copier""",2999.95,5,0.0,1379.977
6521,"""CA-2017-138289""","""16-01-2017""","""18-01-2017""","""Second Class""","""AR-10540""","""Andy Reiter""","""Consumer""","""United States""","""Jackson""","""Michigan""",49201,"""Central""","""OFF-BI-10004995""","""Office Supplies""","""Binders""","""GBC DocuBind P400 Electric Bin…",5443.96,4,0.0,2504.2216
4278,"""US-2016-107440""","""16-04-2016""","""20-04-2016""","""Standard Class""","""BS-11365""","""Bill Shonely""","""Corporate""","""United States""","""Lakewood""","""New Jersey""",8701,"""East""","""TEC-MA-10001047""","""Technology""","""Machines""","""3D Systems Cube Printer, 2nd G…",9099.93,7,0.0,2365.9818


In [ ]:
df.c